In [ ]:
!pip install uv
!uv pip install -r requirements.txt
!pip install geopy
!pip install haversine

In [ ]:
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error
from geopy.distance import great_circle

from datetime import date
import os

print('All imports successful')

In [ ]:
Water_Quality_df       = pd.read_csv('water_quality_training_dataset.csv')
landsat_train_features = pd.read_csv('landsat_features_new.csv')
Terraclimate_df_1      = pd.read_csv('terraclimate_features_new.csv')
Terraclimate_df_2      = pd.read_csv('terraclimate_features_add.csv')
Terraclimate_df_3      = pd.read_csv('terraclimate_features_add2.csv')
Water_Quality_DEM      = pd.read_csv('water_quality_with_dem30.csv')

print('Shapes:')
for name, df in [('WQ', Water_Quality_df), ('Landsat', landsat_train_features),
                  ('TC1', Terraclimate_df_1), ('TC2', Terraclimate_df_2),
                  ('TC3', Terraclimate_df_3), ('DEM', Water_Quality_DEM)]:
    print(f'  {name}: {df.shape}')

In [ ]:
def compute_landsat_indices(df):
    """
    Compute spectral indices from raw Landsat bands.
    Centralised here so training and validation use identical computation.
    """
    eps = 1e-10
    df  = df.copy()

    df['NDVI']  = (df['nir'] - df['red'])     / (df['nir']    + df['red']    + eps)
    df['MNDWI'] = (df['green'] - df['swir16'])/ (df['green']  + df['swir16'] + eps)
    df['NDMI']  = (df['nir'] - df['swir16'])  / (df['nir']    + df['swir16'] + eps)
    df['NDTI']  = (df['red'] - df['green'])   / (df['red']    + df['green']  + eps)
    df['NDBI']   = (df['swir16'] - df['nir'])   / (df['swir16']    + df['nir'] ) 
    df['SAVI']   = ((df['nir'] - df['red'])  / (df['nir']    + df['red'] + 0.5)) * 1.5


# landsat_train_features['NDBI'] = (landsat_train_features['swir16'] - landsat_train_features['nir']) / ((landsat_train_features['swir16'] + landsat_train_features['nir'] + eps))


    for col in ['NDVI','MNDWI','NDMI','NDTI','NDBI','SAVI']:
        df[col] = df[col].astype(float)

    return df

landsat_train_features = compute_landsat_indices(landsat_train_features)
print('Landsat indices computed for training data')

In [ ]:
def combine_datasets(*dfs):
    data = pd.concat(dfs, axis=1)
    data = data.loc[:, ~data.columns.duplicated()]
    return data

wq_data = combine_datasets(
    Water_Quality_df,
    landsat_train_features,
    Terraclimate_df_1,
    Terraclimate_df_2,
    Terraclimate_df_3,
    Water_Quality_DEM
)

wq_data = wq_data.fillna(wq_data.median(numeric_only=True))
print(f'Combined training shape: {wq_data.shape}')
display(wq_data.head(3))

In [ ]:
# Renaming column

wq_data.rename(columns = {'DEM30_Elevation_m' : 'elevation'}, inplace=True)
wq_data.columns

In [ ]:
# Ensuring all the values in correct data type

wq_data.info()

In [ ]:
# Adding temporal features

def temporal_features(df):

    df['Sample Date'] = pd.to_datetime(df['Sample Date'], format='%d-%m-%Y')
    df['Month']       = df['Sample Date'].dt.month
    df['Quarter'] = df['Sample Date'].dt.quarter
    df["Week_of_year"] = df['Sample Date'].dt.isocalendar().week
    
    # Season - South Africa seasons
    season_map = {
        1:'summer', 2:'summer', 12:'summer',
        3:'autumn', 4:'autumn', 5:'autumn',
        6:'winter', 7:'winter', 8:'winter',
        9:'spring', 10:'spring', 11:'spring'
    }
    df['season'] = df['Month'].map(season_map)
    df = pd.get_dummies(df, columns=['season'], dtype=int)
    
    # Ensure all 4 season columns exist
    for s in ['season_summer','season_autumn','season_winter','season_spring']:
        if s not in df.columns:
            df[s] = 0
    
    print('Temporal features added')

    return df

In [ ]:
wq_data = temporal_features(wq_data)

In [ ]:
wq_data['Week_of_year'] = wq_data['Week_of_year'].astype('int')

wq_data.info()

In [ ]:
#wq_data['Day'].value_counts().sort_index()
#wq_data['Month'].value_counts().sort_index()
wq_data['Week_of_year'].value_counts().sort_index()



In [ ]:
def periodic_transform(dff,variable):
    dff[f"{variable}_sin"] = np.sin(dff[variable] / dff[variable].max()*2*np.pi)
    dff[f"{variable}_cos"] = np.cos(dff[variable] / dff[variable].max()*2*np.pi)
    return dff

wq_data = periodic_transform(wq_data, 'Month')
wq_data = periodic_transform(wq_data, 'Week_of_year')


## Spatial Feature Engineering


In [ ]:
# Adding spatial features

SA_cities = {
    'Johannesburg': (-26.205,   28.049722),
    'Cape Town':    (-33.9288,  18.4172),
    #'Durban':       (-29.8618,  31.0099),
    #'Pretoria':     (-25.7459,  28.1879),
    #'Kruger National Park':    (-23.9883,  31.5547),
}

def compute_city_distances(df, cities):
    df = df.copy()
    city_names = list(cities.keys())
    for city, city_coords in cities.items():
        col = 'dist_' + city.lower().replace(' ', '_')
        df[col] = df.apply(
            lambda row: great_circle(
                (row['Latitude'], row['Longitude']),
                city_coords
            ).km, axis=1
        )
    dist_cols = ['dist_' + c.lower().replace(' ', '_') for c in city_names]

    
    #df['dist_nearest_city_km'] = df[dist_cols].min(axis=1)
   
   # Keep only nearest city 
    #cols_to_drop = [c for c in df.columns if 'dist' in c and c != 'dist_nearest_city_km']

    #df = df.drop(cols_to_drop, axis=1)
    
    return df

wq_data = compute_city_distances(wq_data, SA_cities)
print('City distance features added')
print([c for c in wq_data.columns if 'dist' in c])

In [ ]:
for col in wq_data.columns:
    if wq_data[col].dtype == 'float64':
        wq_data[col] = np.round(wq_data[col], 3)

wq_data.head(3)

In [ ]:

wq_data.to_csv('/tmp/complete_data.csv', index=False)

session.sql("""
    PUT file:///tmp/complete_data.csv
    'snow://workspace/USER$.PUBLIC."EY-AI-and-Data-Challenge"/versions/live/'
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

print('Submission saved and uploaded.')
print('Refresh browser to see file in sidebar.')

# Validation data Preparation


In [ ]:
test_file = pd.read_csv("submission_template.csv")
landsat_val_features = pd.read_csv("landsat_features_validation_new.csv")
Terraclimate_val_df_1 = pd.read_csv("terraclimate_features_validation_new.csv")
Terraclimate_val_df_2 = pd.read_csv("terraclimate_features_validation_add.csv")
Terraclimate_val_df_3 = pd.read_csv("terraclimate_features_validation_add2.csv")
Water_Quality_DEM_val = pd.read_csv('submission_template_with_dem.csv')


print('Shapes:')
for name, df in [('WQ', test_file), ('Landsat', landsat_val_features),
                  ('TC1', Terraclimate_val_df_1), ('TC2', Terraclimate_val_df_2),
                  ('TC3', Terraclimate_val_df_3), ('DEM', Water_Quality_DEM_val)]:
    print(f'  {name}: {df.shape}')

In [ ]:
landsat_val_features = compute_landsat_indices(landsat_val_features)
print('Landsat indices computed for training data')

In [ ]:
df_val = combine_datasets(
    test_file,
    landsat_val_features,
    Terraclimate_val_df_1,
    Terraclimate_val_df_2,
    Terraclimate_val_df_3,
    Water_Quality_DEM_val
)

df_val = df_val.fillna(df_val.median(numeric_only=True))
print(f'Combined training shape: {df_val.shape}')
display(df_val.head(3))

In [ ]:
# Renaming column

df_val.rename(columns = {'DEM30_Elevation_m' : 'elevation'}, inplace=True)
df_val = temporal_features(df_val)
df_val = periodic_transform(df_val, 'Month')
df_val = periodic_transform(df_val, 'Week_of_year')
df_val.columns

In [ ]:
df_val = compute_city_distances(df_val, SA_cities)
print('City distance features added')
print([c for c in df_val.columns if 'dist' in c])




In [ ]:
df_val.shape

In [ ]:
for col in df_val.columns:
    if df_val[col].dtype == 'float64':
        df_val[col] = np.round(df_val[col], 3)

df_val.head(3)

In [ ]:
df_val.describe()

In [ ]:

df_val.to_csv('/tmp/complete_val_data.csv', index=False)

session.sql("""
    PUT file:///tmp/complete_val_data.csv
    'snow://workspace/USER$.PUBLIC."EY-AI-and-Data-Challenge"/versions/live/'
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

print('Submission saved and uploaded.')
print('Refresh browser to see file in sidebar.')